# Simple DP2 Pipeline Demo

Load one DP2 patch, inject 100 clusters per run, detect them with MCI, and compare recovery across bands and batch rounds. A selected-band example also uses Rubin's native injector to preserve variance and mask updates on a copy of the exposure.

Edit the settings, then run top to bottom in your CANFAR Rubin kernel. Batch and all-band runs use the simple array injection path; only the selected-band native example updates the Rubin variance plane.

## 1. Imports
Run this in the Rubin kernel where INJECT is installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from inject import ClusterConfig, InjectionConfig, InjectionPipeline, notebook_output_dir

## 2. Settings
Set your DP2 data ID, bands and run sizes here. This example assumes calibrated DP2 nJy images.

In [ ]:
BUTLER_REPO = 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml'
BUTLER_COLLECTIONS = 'dp2'
DATA_ID = {'skymap': 'lsst_cells_v2', 'tract': 7508, 'patch': 0}
BANDS = ['g', 'r', 'i']
SELECTED_BAND = 'r'
N_CLUSTERS = 100
N_ROUNDS = 10
OUTPUT_DIR = notebook_output_dir('simple_dp2_pipeline_demo')

config = InjectionConfig(
    bands=BANDS, band=SELECTED_BAND, n_clusters=N_CLUSTERS,
    cutout_size=800, edge_buffer=60, seed=42,
    use_actual_psf=True, add_noise=False, pixel_scale=0.2, zero_point=31.4,
    output_dir=str(OUTPUT_DIR / 'selected'),
    cluster_config=ClusterConfig(
        profile_type='king', mag_min=20.5, mag_max=24.5,
        r_half_min=2.0, r_half_max=6.0,
        concentration_min=5.0, concentration_max=20.0,
    ),
)

## 3. Load DP2
Load each band with Butler. INJECT converts DP2 cell coadds and keeps the selected band's native product for the Rubin variance-aware example.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(
    butler=butler, data_id=DATA_ID, dataset_type='deep_coadd',
    native_bands=[SELECTED_BAND],
)

print({band: image.shape for band, image in pipe.images.items()})

## 4. Simple MCI detection
The built-in MCI detector returns pixel-coordinate detections. `pipe.analyze()` matches them one-to-one to injected sources within `MATCH_RADIUS` pixels. A background source can still match by chance; this quick demo does not subtract detections from the original image.

In [ ]:
import pandas as pd
from inject import ClusterRetrieval
from inject.detection import run_cluster_detection

MATCH_RADIUS = 5.0
DETECTOR_PSF_FWHM = 3.5

def detector(image):
    return run_cluster_detection(
        image, psf_fwhm=DETECTOR_PSF_FWHM, pixel_scale=config.pixel_scale,
        threshold_sigma=3.0, snr_min=3.0, mci_max=0.85,
        r_half_min=1.0, ellipticity_max=0.6, box_size=64,
        use_multiscale=True, use_mci=True, verbose=False,
    )

In [ ]:
baseline_detections = {band: detector(pipe.images[band]) for band in BANDS}

### Plot helper
Use INJECT's `make_plots()` for before/after, recovered/missed positions, 1D/2D completeness, PSF width, and postage stamps.

In [ ]:
from dataclasses import replace
from inject.plotting import plot_completeness_1d, plot_completeness_2d

def plot_run(band, result, folder):
    folder = OUTPUT_DIR / folder
    view = InjectionPipeline(replace(config, band=band, bands=None, output_dir=str(folder)))
    view.load_data(image=pipe.images[band])
    view.injected_image = result['injected_image']
    view.injection_info = result['injection_info']
    view.retrieval = ClusterRetrieval.from_matched(view.injection_info)
    view.make_plots(output_dir=str(folder), show=True, save=True, n_stamps=3)
    vmin, vmax = np.nanpercentile(view.image, [1, 99])
    for name, image in [('original', view.image), ('injected', view.injected_image)]:
        plt.imsave(folder / f'{name}.png', image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    plt.close('all')

## 5. All bands
Inject and detect in each band with `run_batch_multiband()`. Show one RGB image and save per-band completeness plots.

In [ ]:
from astropy.visualization import make_lupton_rgb

RGB_BANDS = ('i', 'r', 'g')
if not set(RGB_BANDS).issubset(BANDS):
    raise ValueError('Keep g, r and i in BANDS for the colour image.')
if len({pipe.images[band].shape for band in RGB_BANDS}) != 1 or len({pipe.bboxes[band] for band in RGB_BANDS}) != 1:
    raise ValueError('The colour image needs matching band cutouts.')

all_bands = pipe.run_batch_multiband(
    n_iterations=1, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'all_bands'),
    store_images=True, use_psf_cache=False, n_workers=1, verbose=False,
)

all_recovery = []
for band in BANDS:
    result = all_bands[band][0]
    rows = result['injection_info']
    assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
    retrieval = ClusterRetrieval(rows, result['detections'])
    retrieval.match_detections(
        match_radius=MATCH_RADIUS, baseline_detections=baseline_detections[band],
    )
    all_recovery.append({'band': band, **retrieval.get_summary_statistics()})

channels = [np.nan_to_num(all_bands[band][0]['injected_image'] - np.nanmedian(pipe.images[band]),
                          nan=0.0, posinf=0.0, neginf=0.0) for band in RGB_BANDS]
original_channels = [np.nan_to_num(pipe.images[band] - np.nanmedian(pipe.images[band]),
                                   nan=0.0, posinf=0.0, neginf=0.0) for band in RGB_BANDS]
stretch = max(float(np.percentile(np.abs(channels), 99.5)), 1e-6)
rgb = make_lupton_rgb(*channels, stretch=stretch, Q=8)
original_rgb = make_lupton_rgb(*original_channels, stretch=stretch, Q=8)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for axis, image, title in zip(axes, [original_rgb, rgb, rgb],
                              ['Original RGB', 'Injected RGB', f'Recovery status in {SELECTED_BAND}']):
    axis.imshow(image, origin='lower')
    axis.set_title(title)
    axis.set_axis_off()
status_classes = [
    (True, False, 'lime', 'Clean recovery'),
    (False, True, 'orange', 'Background-confounded'),
    (False, False, 'red', 'Missed'),
]
rows = all_bands[SELECTED_BAND][0]['injection_info']
for detected, confounded, color, label in status_classes:
    points = [row for row in rows
              if row.get('background_confounded', False) == confounded
              and row['detected'] == detected]
    if points:
        axes[2].scatter([row['x'] for row in points], [row['y'] for row in points],
                        facecolors='none', edgecolors=color, s=35, label=label)
axes[2].legend()
fig.savefig(OUTPUT_DIR / 'all_bands_rgb.png', dpi=150, bbox_inches='tight')
plt.show()
plt.imsave(OUTPUT_DIR / 'original_rgb.png', original_rgb, origin='lower')
plt.imsave(OUTPUT_DIR / 'injected_rgb.png', rgb, origin='lower')

all_recovery = pd.DataFrame(all_recovery)
all_recovery.to_csv(OUTPUT_DIR / 'all_bands_recovery.csv', index=False)
for band in BANDS:
    plot_run(band, all_bands[band][0], f'all_bands/{band}/plots')
all_recovery[['band', 'n_injected', 'n_detected', 'n_background_confounded',
              'clean_completeness', 'overall_completeness']]

## 6. One band
Use `generate_catalog()`, `inject()`, `detect_with()` and `analyze()` for one selected-band run.

In [ ]:
pipe.image = pipe.images[SELECTED_BAND]
selected_catalog = pipe.generate_catalog()
injected_image, injection_info = pipe.inject(
    catalog=selected_catalog, band=SELECTED_BAND, verbose=False,
)
assert injection_info and all(row['psf_used'] == 'rubin' for row in injection_info), 'Check Rubin PSF evaluation.'
pipe.detect_with(detector)
selected_recovery = pipe.analyze(
    match_radius=MATCH_RADIUS,
    baseline_detections=baseline_detections[SELECTED_BAND],
)
pipe.save_results()

selected_result = {'injected_image': injected_image, 'injection_info': injection_info}
plot_run(SELECTED_BAND, selected_result, 'selected/plots')
pd.DataFrame([{'band': SELECTED_BAND, **selected_recovery}])[
    ['band', 'n_injected', 'n_detected', 'n_background_confounded',
     'clean_completeness', 'overall_completeness']
]

## 7. Rubin native injection
Repeat the same catalog with `inject_with_rubin()`. It clones the original exposure and updates variance and injection-mask planes on the copy. MCI still estimates noise from the image. Compare the saved variance and mask outputs.

In [ ]:
original = pipe.native_exposures[SELECTED_BAND]
original_variance = pipe.variances[SELECTED_BAND]
original_mask = pipe.masks[SELECTED_BAND]

native_image, native_info = pipe.inject_with_rubin(
    catalog=selected_catalog, band=SELECTED_BAND, add_noise=False,
)
native_detections = detector(native_image)
native_retrieval = ClusterRetrieval(native_info, native_detections)
native_retrieval.match_detections(
    match_radius=MATCH_RADIUS,
    baseline_detections=baseline_detections[SELECTED_BAND],
)
plot_run(SELECTED_BAND, {
    'injected_image': native_image, 'injection_info': native_retrieval._matched,
}, 'selected/rubin_native/plots')

assert np.array_equal(original.variance.array[:config.cutout_size, :config.cutout_size], original_variance)
assert np.array_equal(original.mask.array[:config.cutout_size, :config.cutout_size], original_mask)
print('Injected:', pipe.native_injection_summary)
print('Changed variance pixels:', np.count_nonzero(pipe.native_variance_delta))
print('Changed mask pixels:', np.count_nonzero(pipe.native_mask_changed))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for axis, image, title in zip(
    axes, (original_variance, pipe.injected_variance, pipe.native_variance_delta),
    ('Original variance', 'Injected variance', 'Added variance'),
):
    axis.imshow(image, origin='lower', cmap='magma')
    axis.set_title(title)
    axis.set_axis_off()
fig.savefig(OUTPUT_DIR / 'selected' / 'rubin_native' / 'variance_planes.png', dpi=150)
plt.show()
plt.imsave(OUTPUT_DIR / 'selected' / 'rubin_native' / 'mask_changed.png', pipe.native_mask_changed, origin='lower', cmap='gray')

## 8. Batch
Run ten rounds of 100 clusters per band. Recovery is matched within each round; pooled completeness plots combine the already matched results.

In [ ]:
PLOT_ALL_ROUNDS = False
batch = pipe.run_batch_multiband(
    n_iterations=N_ROUNDS, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'batch'),
    store_images=True, use_psf_cache=False, n_workers=1, verbose=False,
)

batch_recovery = []
for band, rounds in batch.items():
    for result in rounds:
        rows = result['injection_info']
        assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
        retrieval = ClusterRetrieval(rows, result['detections'])
        retrieval.match_detections(
            match_radius=MATCH_RADIUS,
            baseline_detections=baseline_detections[band],
        )
        batch_recovery.append({
            'band': band, 'round': result['iteration'] + 1, **retrieval.get_summary_statistics(),
        })
    for result in (rounds if PLOT_ALL_ROUNDS else rounds[:1]):
        plot_run(band, result, f"batch/{band}/plots/round_{result['iteration'] + 1:02d}")
    matched_rows = [row for result in rounds for row in result['injection_info']]
    pooled = ClusterRetrieval.from_matched(matched_rows)
    folder = OUTPUT_DIR / 'batch' / band / 'plots'
    for name, plotter in [('completeness_1d', plot_completeness_1d), ('completeness_2d', plot_completeness_2d)]:
        fig, _ = plotter(pooled, config)
        fig.suptitle(f'{band} band: all {len(rounds)} rounds')
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        fig.savefig(folder / f'{name}_all_rounds.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
    for result in rounds:
        result.pop('injected_image', None)

batch_recovery = pd.DataFrame(batch_recovery)
batch_recovery.to_csv(OUTPUT_DIR / 'batch_recovery.csv', index=False)
batch_totals = batch_recovery.groupby('band')[
    ['n_injected', 'n_detected', 'n_background_confounded', 'n_clean_eligible']
].sum()
batch_totals['overall_completeness'] = batch_totals['n_detected'] / batch_totals['n_injected']
batch_totals['clean_completeness'] = batch_totals['n_detected'] / batch_totals['n_clean_eligible']
print('Outputs:', OUTPUT_DIR)
batch_totals